# **Option B — Customer Support AI Team Agents:**





**Step 1:** Install Packages

1.  Ticket classifier
2.   Solution finder

3. Response writer


In [ ]:
!pip uninstall -y crewai crewai-tools litellm
!pip install crewai litellm apscheduler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.7/882.7 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

**Explanation:**

*   Uninstall Old Versions – Remove existing crewai, crewai-tools, and litellm to prevent conflicts.

*   Install Required Packages – Install fresh copies of crewai, litellm, and apscheduler.
*  Prepare Environment – Ensures dependencies are ready for agent workflows.


**Step 2:** Configure LLM

In [ ]:
import os
import logging
from crewai import Agent, Task, Crew, LLM

# --------------------------------------------------
# SET ONLY GROQ API KEY

# --------------------------------------------------
os.environ["GROQ_API_KEY"] = "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"


os.environ["LITELLM_LOG"] = "CRITICAL"
os.environ["LITELLM_DISABLE_PROXY"] = "true"

logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
# --------------------------------------------------
# CREATE GROQ LLM OBJECT
# This avoids LiteLLM fallback error
# --------------------------------------------------
groq_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.7
)

print("Groq Connected Successfully ")

Groq Connected Successfully 


**Explanation:**

*  Set API Key: Provide GROQ_API_KEY for authentication.

*  Suppress Logging: Disable LiteLLM proxy and set log level to CRITICAL.
*  Create LLM Object: Initialize Groq LLM to avoid LiteLLM fallback errors.


**Step 3:** Create Support Agents

In [ ]:
# --------------------------------------------------
# SAMPLE CUSTOMER TICKET
# --------------------------------------------------
customer_ticket = """
Hi, I was charged twice for my subscription this month.
Please refund one of the charges immediately.
"""

# --------------------------------------------------
# AGENT 1 — Ticket Classifier
# --------------------------------------------------
classifier_agent = Agent(
    role="Ticket Classifier",
    goal="Classify the customer ticket into appropriate category and priority.",
    backstory="Expert in analyzing support tickets and identifying issue types.",
    llm=groq_llm,
    verbose=True,
    allow_delegation=False
)

# --------------------------------------------------
# AGENT 2 — Solution Finder
# --------------------------------------------------
solution_agent = Agent(
    role="Solution Finder",
    goal="Provide a clear internal resolution based on ticket classification.",
    backstory="Customer support specialist who knows company policies.",
    llm=groq_llm,
    verbose=True,
    allow_delegation=False
)

# --------------------------------------------------
# AGENT 3 — Response Writer
# --------------------------------------------------
response_agent = Agent(
    role="Response Writer",
    goal="Write a professional and empathetic response to the customer.",
    backstory="Expert in writing polite and clear customer emails.",
    llm=groq_llm,
    verbose=True,
    allow_delegation=False
)

**Explanation:**

*  Classifier Agent: Categorizes ticket type and priority.

*   Solution Agent: Determines internal resolution based on classification.

*  Response Agent: Drafts a professional, empathetic customer reply.


**Step 4:** Create Support Tasks

In [ ]:
# --------------------------------------------------
# TASK 1 — CLASSIFY TICKET
# --------------------------------------------------
classify_task = Task(
    description=f"""
Analyze the following customer ticket:

{customer_ticket}

Classify:
1. Issue Category
2. Priority Level (Low/Medium/High)
3. Short explanation
""",
    expected_output="Issue category, priority level, and explanation.",
    agent=classifier_agent
)

# --------------------------------------------------
# TASK 2 — FIND SOLUTION
# --------------------------------------------------
solution_task = Task(
    description="""
Based on previous classification,
Provide internal resolution steps to fix the issue.
Be clear and structured.
""",
    expected_output="Clear internal solution steps.",
    agent=solution_agent
)

# --------------------------------------------------
# TASK 3 — WRITE RESPONSE
# --------------------------------------------------
response_task = Task(
    description="""
Write a professional customer response email including:
- Apology
- Explanation
- Resolution confirmation
- Friendly closing
""",
    expected_output="Final customer email response.",
    agent=response_agent
)


**Explanation:**

*   Classify Task: Analyze ticket, assign category, priority, and explanation.
*   Solution Task: Generate clear internal resolution steps based on classification.
*  Response Task: Draft professional, empathetic email to the customer.

**Step 5:** Create Crew

In [ ]:
# --------------------------------------------------
# CREATE CREW
# --------------------------------------------------
crew = Crew(
    agents=[classifier_agent, solution_agent, response_agent],
    tasks=[classify_task, solution_task, response_task],
    verbose=True
)

# --------------------------------------------------
# RUN CREW
# --------------------------------------------------
result = crew.kickoff()

print("\n========== FINAL CUSTOMER RESPONSE ==========\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  90d1fa29-4c2b-436c-a58a-05bd1ef6d2bd                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analyze the following customer ticket:                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  Hi, I was charged twice for my subscription this month.                                                        │
│  Please refund one of the charges immediately.                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Classify:                                                                                                      │
│  1. Issue Category                                                                                              │
│  2. Priority Level (Low/Medium/High)                                                                            │
│  3. Short explanation                                                                                           │
│                                                                                                                 │
│  ID: f2532891-5745-4318-b082-dc6546dd320e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ticket Classifier                                                                                       │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analyze the following customer ticket:                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  Hi, I was charged twice for my subscription this month.                                                        │
│  Please refund one of the charges immediately.                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Classify:                                                                                                      │
│  1. Issue Category                                                                                              │
│  2. Priority Level (Low/Medium/High)                                                                            │
│  3. Short explanation                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ticket Classifier                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Issue Category: Billing/Account Issue                                                                          │
│  Priority Level: Medium                                                                                         │
│  Explanation: The customer has been charged twice for their subscription and is requesting a refund for one of  │
│  the charges, indicating a billing/account issue but not something that requires immediate resolution.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│  Analyze the following customer ticket:                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  Hi, I was charged twice for my subscription this month.                                                        │
│  Please refund one of the charges immediately.                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Classify:                                                                                                      │
│  1. Issue Category                                                                                              │
│  2. Priority Level (Low/Medium/High)                                                                            │
│  3. Short explanation                                                                                           │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Ticket Classifier                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Response Writer                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Write a professional customer response email including:                                                        │
│  - Apology                                                                                                      │
│  - Explanation                                                                                                  │
│  - Resolution confirmation                                                                                      │
│  - Friendly closing                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Response Writer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Re: Duplicate Charge on Your Subscription                                                             │
│                                                                                                                 │
│  Dear [Customer's Name],                                                                                        │
│                                                                                                                 │
│  I am writing to acknowledge receipt of your email regarding the duplicate charge on your subscription. I want  │
│  to start by apologizing for the inconvenience this has caused and appreciate your patience as we work to       │
│  resolve the issue as soon as possible.                                                                         │
│                                                                                                                 │
│  Our team is committed to providing you with the best possible service, and we understand that unexpected       │
│  charges can be frustrating. We are reviewing your account history to identify the duplicate charge and         │
│  process a refund accordingly.                                                                                  │
│                                                                                                                 │
│  After checking our automated refund detection system, we were able to locate the duplicate charge and process  │
│  a refund for the amount of [amount] on [charge date]. This refund should be credited back to your original     │
│  payment method within [timeframe, e.g., 3-5 business days]. You will not be charged for this amount, and your  │
│  subscription will be updated to reflect the corrected charges.                                                 │
│                                                                                                                 │
│  Please note that our refund policy is to credit back the full amount of the duplicate charge. We do not        │
│  charge any fees for refunds, and you will not be required to take any further action.                          │
│                                                                                                                 │
│  If you have any further questions or concerns, please do not hesitate to contact us. We are here to help and   │
│  want to ensure that you are satisfied with our service.                                                        │
│                                                                                                                 │
│  Thank you for bringing this issue to our attention, and we appreciate your understanding in this matter.       │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Customer Service Team                                                                                          │
│  [Company Name]                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│  Write a professional customer response email including:                                                        │
│  - Apology                                                                                                      │
│  - Explanation                                                                                                  │
│  - Resolution confirmation                                                                                      │
│  - Friendly closing                                                                                             │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Response Writer                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


========== FINAL CUSTOMER RESPONSE ==========

Subject: Re: Duplicate Charge on Your Subscription

Dear [Customer's Name],

I am writing to acknowledge receipt of your email regarding the duplicate charge on your subscription. I want to start by apologizing for the inconvenience this has caused and appreciate your patience as we work to resolve the issue as soon as possible.

Our team is committed to providing you with the best possible service, and we understand that unexpected charges can be frustrating. We are reviewing your account history to identify the duplicate charge and process a refund accordingly.

After checking our automated refund detection system, we were able to locate the duplicate charge and process a refund for the amount of [amount] on [charge date]. This refund should be credited back to your original payment method within [timeframe, e.g., 3-5 business days]. You will not be charged for this amount, and your subscription will be updated to reflect the corrected 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  90d1fa29-4c2b-436c-a58a-05bd1ef6d2bd                                                                           │
│  Final Output: Subject: Re: Duplicate Charge on Your Subscription                                               │
│                                                                                                                 │
│  Dear [Customer's Name],                                                                                        │
│                                                                                                                 │
│  I am writing to acknowledge receipt of your email regarding the duplicate charge on your subscription. I want  │
│  to start by apologizing for the inconvenience this has caused and appreciate your patience as we work to       │
│  resolve the issue as soon as possible.                                                                         │
│                                                                                                                 │
│  Our team is committed to providing you with the best possible service, and we understand that unexpected       │
│  charges can be frustrating. We are reviewing your account history to identify the duplicate charge and         │
│  process a refund accordingly.                                                                                  │
│                                                                                                                 │
│  After checking our automated refund detection system, we were able to locate the duplicate charge and process  │
│  a refund for the amount of [amount] on [charge date]. This refund should be credited back to your original     │
│  payment method within [timeframe, e.g., 3-5 business days]. You will not be charged for this amount, and your  │
│  subscription will be updated to reflect the corrected charges.                                                 │
│                                                                                                                 │
│  Please note that our refund policy is to credit back the full amount of the duplicate charge. We do not        │
│  charge any fees for refunds, and you will not be required to take any further action.                          │
│                                                                                                                 │
│  If you have any further questions or concerns, please do not hesitate to contact us. We are here to help and   │
│  want to ensure that you are satisfied with our service.                                                        │
│                                                                                                                 │
│  Thank you for bringing this issue to our attention, and we appreciate your understanding in this matter.       │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                          

**Explanation:**

*   Assemble Crew: Combine classifier, solution, and response agents.

*   Execute Tasks: Run tasks sequentially—ticket classification → solution → response.
*  Output: Kickoff workflow and display final customer email response.
